# Assessed project - worked solution (Option A)

**Predicting chromatographic retention from molecular descriptors**
Example authorship: instructor reference solution.

This is a reference solution for Option A, showing the standard expected of a
full-mark submission: a leakage-free grouped split, a baseline plus an improved
model, cross-validation with an uncertainty estimate, interpretation, and a
completed LLM-use appendix. The data are ILLUSTRATIVE and generated in-notebook so
the solution runs anywhere; a real submission would use measured data (e.g. via
DeepLC or a public deposition) but the workflow is identical.

## 0. Setup

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score
RANDOM_STATE=42; rng=np.random.default_rng(RANDOM_STATE)

## 1. Problem and data description
Task: predict chromatographic retention time (minutes) from five molecular
descriptors (logP, molecular weight, TPSA, H-bond donors, H-bond acceptors), to
filter candidate identifications. Data: illustrative, generated below (synthetic
but realistic). Licence: not applicable (synthetic); a real dataset's licence
would be stated here.

In [ ]:
n_mol=140
logP=rng.normal(2.5,1.5,n_mol)
mw=rng.normal(300,90,n_mol).clip(120,650)
tpsa=rng.normal(75,35,n_mol).clip(5,200)
hbd=rng.integers(0,6,n_mol); hba=rng.integers(1,11,n_mol)
rt_true=2+3.2*logP+0.012*mw-0.03*tpsa+0.2*hba
rows=[]
for i in range(n_mol):
    for _ in range(2):  # 2 replicate injections, shared mol_id
        rows.append([i,logP[i],mw[i],tpsa[i],hbd[i],hba[i],max(0.5,rt_true[i]+rng.normal(0,0.4))])
df=pd.DataFrame(rows,columns=["mol_id","logP","mw","tpsa","hbd","hba","retention_min"])
print(df.shape); df.head()

## 2. Exploratory analysis

In [ ]:
print(df.describe().round(2))
plt.scatter(df.logP,df.retention_min,alpha=0.6,color="#028090")
plt.xlabel("logP"); plt.ylabel("retention (min)"); plt.title("Retention vs logP"); plt.show()

## 3. Preprocessing and splitting
Two replicate injections share a `mol_id`. We split BY molecule with
GroupShuffleSplit so no compound is in both train and test; otherwise the model
would recognise compounds rather than predict (leakage, Part 3).

In [ ]:
features=["logP","mw","tpsa","hbd","hba"]
X=df[features].values; y=df["retention_min"].values; groups=df["mol_id"].values
tr,te=next(GroupShuffleSplit(n_splits=1,test_size=0.3,random_state=RANDOM_STATE).split(X,y,groups))
X_tr,X_te,y_tr,y_te=X[tr],X[te],y[tr],y[te]
print("shared compounds between train and test:",len(set(groups[tr])&set(groups[te])))

## 4. Modelling: baseline + improved

In [ ]:
base=LinearRegression().fit(X_tr,y_tr)
rf=RandomForestRegressor(n_estimators=400,random_state=RANDOM_STATE).fit(X_tr,y_tr)
print("baseline  test MAE:",round(mean_absolute_error(y_te,base.predict(X_te)),2),"min")
print("random forest MAE :",round(mean_absolute_error(y_te,rf.predict(X_te)),2),"min")

## 5. Evaluation: grouped cross-validation + uncertainty
We cross-validate with GroupKFold (groups kept whole) and report the mean and
standard deviation across folds as an uncertainty on the estimate.

In [ ]:
scores=cross_val_score(RandomForestRegressor(n_estimators=400,random_state=RANDOM_STATE),
                       X,y,groups=groups,cv=GroupKFold(5),scoring="r2")
print("grouped CV R2: %.3f +/- %.3f"%(scores.mean(),scores.std()))
print("test R2 (held-out compounds):",round(r2_score(y_te,rf.predict(X_te)),3))

## 6. Interpretation

In [ ]:
imp=pd.Series(rf.feature_importances_,index=features).sort_values()
imp.plot.barh(color="#028090"); plt.title("Feature importances"); plt.xlabel("importance"); plt.show()
print(imp.round(3))
print("\nThe model leans on logP and molecular weight, matching how retention was constructed - a sanity check passes.")

## 7. LLM-use appendix (completed example)

**Use 1 - scaffolding the grouped split.**
Prompt: "In scikit-learn, how do I split a dataset so that all rows sharing a
group id stay on the same side of a train/test split?" The model suggested
GroupShuffleSplit, which was correct. Verified by printing the number of shared
compounds between train and test (0).

**Use 2 - uncertainty estimate.**
Prompt: "How can I report an uncertainty on a cross-validated R2 in scikit-learn?"
The model suggested the standard deviation across folds, which is reasonable. It
initially proposed plain KFold; I changed it to GroupKFold to avoid replicate
leakage, which the model had not considered.

**Use 3 - plotting.**
Prompt: "Plot feature importances as a horizontal bar chart from a fitted
RandomForestRegressor." Correct; verified visually.

## 8. Conclusion
A random forest predicts retention with a low MAE and clearly beats the linear
baseline, with a stable grouped-CV R2. The model relies on chemically sensible
descriptors. The main limitation is that the data here are illustrative; on real
data the same pipeline, with the grouped split kept, would give a trustworthy
estimate. The key safeguard is the by-molecule split: an ordinary random split
would have inflated the score by letting replicates leak.